### The aims of this project is to import data from the ign(csv file) into a database an set correct datatype for each column on the database

In [1]:
import psycopg2
import csv
import pandas as pd

conn = psycopg2.connect(dbname="guyzo", user="guyzo")
cur = conn.cursor()

#### Creating the database with default column (without optimizing)

In [2]:
sql_query = """
    CREATE TABLE IF NOT EXISTS ign_reviews(
    id 	integer PRIMARY KEY,
    score_phrase text,
    title text,
    url text,
    platform text,
 	score real,
 	genre text,
 	editors_choice text,
 	release_year integer,
 	release_month integer,
 	release_day int);
"""

cur.execute(sql_query)

conn.commit()

### Displaying the schema information        

In [3]:
sql_query = """
SELECT 
    column_name,
    data_type,
    is_nullable,
    column_default
FROM information_schema.columns
WHERE table_name = 'ign_reviews'
ORDER BY ordinal_position;
"""

cur.execute(sql_query)

rows = cur.fetchall()

# getting columns name
columns = [desc[0] for desc in cur.description]

# creating the dataframe
df_schema = pd.DataFrame(rows, columns=columns)

# Render result as a dataframe
display(df_schema)

,column_name,data_type,is_nullable,column_default
0,id,integer,NO,None
1,score_phrase,text,YES,None
2,title,text,YES,None
3,url,text,YES,None
4,platform,text,YES,None
5,score,real,YES,None
6,genre,text,YES,None
7,editors_choice,text,YES,None
8,release_year,integer,YES,None
9,release_month,integer,YES,None


#### Let's try to import data into our database

In [4]:
with open('ign.csv', 'r') as file:
    next(file)
    reader = csv.reader(file)
    for row in reader:
        cur.execute("INSERT INTO ign_reviews VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s);", row)

conn.commit()

In [5]:
cur.execute("""SELECT * FROM ign_reviews LIMIT 10;""")

rows = cur.fetchall()

columns=[col[0] for col in cur.description]

df_select = pd.DataFrame(rows, columns=columns)

display(df_select)

,id,score_phrase,title,url,platform,score,genre,editors_choice,release_year,release_month,release_day
0,0,Amazing,LittleBigPlanet PS Vita,/games/littlebigplanet-vita/vita-98907,PlayStation Vita,9.0,Platformer,Y,2012,9,12
1,1,Amazing,LittleBigPlanet PS Vita -- Marvel Super Hero E...,/games/littlebigplanet-ps-vita-marvel-super-he...,PlayStation Vita,9.0,Platformer,Y,2012,9,12
2,2,Great,Splice: Tree of Life,/games/splice/ipad-141070,iPad,8.5,Puzzle,N,2012,9,12
3,3,Great,NHL 13,/games/nhl-13/xbox-360-128182,Xbox 360,8.5,Sports,N,2012,9,11
4,4,Great,NHL 13,/games/nhl-13/ps3-128181,PlayStation 3,8.5,Sports,N,2012,9,11
5,5,Good,Total War Battles: Shogun,/games/total-war-battles-shogun/mac-142565,Macintosh,7.0,Strategy,N,2012,9,11
6,6,Awful,Double Dragon: Neon,/games/double-dragon-neon/xbox-360-131320,Xbox 360,3.0,Fighting,N,2012,9,11
7,7,Amazing,Guild Wars 2,/games/guild-wars-2/pc-896298,PC,9.0,RPG,Y,2012,9,11
8,8,Awful,Double Dragon: Neon,/games/double-dragon-neon/ps3-131321,PlayStation 3,3.0,Fighting,N,2012,9,11
9,9,Good,Total War Battles: Shogun,/games/total-war-battles-shogun/pc-142564,PC,7.0,Strategy,N,2012,9,11


#### Let's analyze the score column

In [6]:
cur.execute("""SELECT DISTINCT score FROM ign_reviews ORDER BY 1 ASC;""")

rows = cur.fetchall()
print(rows)

[(0.5,), (0.7,), (0.8,), (1.0,), (1.1,), (1.2,), (1.3,), (1.4,), (1.5,), (1.7,), (1.8,), (1.9,), (2.0,), (2.1,), (2.2,), (2.3,), (2.4,), (2.5,), (2.6,), (2.7,), (2.8,), (2.9,), (3.0,), (3.1,), (3.2,), (3.3,), (3.4,), (3.5,), (3.6,), (3.7,), (3.8,), (3.9,), (4.0,), (4.1,), (4.2,), (4.3,), (4.4,), (4.5,), (4.6,), (4.7,), (4.8,), (4.9,), (5.0,), (5.1,), (5.2,), (5.3,), (5.4,), (5.5,), (5.6,), (5.7,), (5.8,), (5.9,), (6.0,), (6.1,), (6.2,), (6.3,), (6.4,), (6.5,), (6.6,), (6.7,), (6.8,), (6.9,), (7.0,), (7.1,), (7.2,), (7.3,), (7.4,), (7.5,), (7.6,), (7.7,), (7.8,), (7.9,), (8.0,), (8.1,), (8.2,), (8.3,), (8.4,), (8.5,), (8.6,), (8.7,), (8.8,), (8.9,), (9.0,), (9.1,), (9.2,), (9.3,), (9.4,), (9.5,), (9.6,), (9.7,), (9.8,), (9.9,), (10.0,)]


We observe that the `score` column contains decimal values with a maximum of three digits. Based on this observation, we can use the `NUMERIC` data type to store the values with exact decimal precision.

In [7]:
sql_query="""
    ALTER TABLE ign_reviews
    ALTER COLUMN score TYPE NUMERIC(3,1);
"""

cur.execute(sql_query)
conn.commit()


In [8]:
sql_query = """
SELECT 
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'ign_reviews'  AND column_name='score'
ORDER BY ordinal_position;
"""

cur.execute(sql_query)

rows = cur.fetchall()

# getting columns name
columns = [desc[0] for desc in cur.description]

# creating the dataframe
df_schema = pd.DataFrame(rows, columns=columns)

# Render result as a dataframe
display(df_schema)

,column_name,data_type
0,score,numeric


#### Let's analyse the score_phrase column

In [9]:
import csv
with open('ign.csv', 'r') as f:
    next(f) # skip the row containing column headers
    reader = csv.reader(f)
    # create a set to contain all score phrases
    unique_words_in_score_phrase = set()
    for row in reader:
        # add the score phrase from this row to the set
        score_phrase = row[1]
        unique_words_in_score_phrase.add(score_phrase)
max_len = 0

for score_phrase in unique_words_in_score_phrase:
    max_len = max(max_len, len(score_phrase))
print(max_len)

11


The `score_phrase` column is currently stored as `TEXT`. After analyzing the values in this column, we found that no value requires more than 11 characters. Therefore, we can change its data type to `VARCHAR(11)` to reduce the storage size required by the database while enforcing a maximum length of 11 characters.

In [10]:
sql_query="""
    ALTER TABLE ign_reviews
    ALTER COLUMN score_phrase TYPE varchar(11)
"""

cur.execute(sql_query)

conn.commit()

The `score_phrase` column contains a limited and predefined set of values. Although `VARCHAR(11)` restricts the maximum length, it would still allow invalid values such as `Dataquest`. Therefore, an enumerated type is a better solution because it restricts the column to the valid values only. PostgreSQL internally represents each enum value using a 4-byte identifier, reducing the storage required compared with storing the full string.

In [11]:
# Create ENUM type
cur.execute("""
    CREATE TYPE evaluation_enum AS ENUM(
        'Great', 'Mediocre', 'Bad', 'Good', 
    'Awful', 'Okay', 'Masterpiece', 'Amazing', 
    'Unbearable', 'Disaster', 'Painful');
""")

# Commit change to create the type evaluation_enum
conn.commit()

In [12]:
# Adding evaluation_enum to score_phrase column

cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN score_phrase TYPE evaluation_enum
    USING score_phrase::evaluation_enum;
""")

# Commit changes
conn.commit()
    

 Let's make same change for the `genre` and `platform` column

In [13]:
# cur.execute("""
#    CREATE TYPE genre_enum AS ENUM(
#        'Adventure', 'Strategy', 'Shooter', 'genre', 'Virtual Pet', 'Hardware', 'Adult', 'Baseball', 
#        'Sports', 'Flight', 'Unknown', 'Racing', 'Battle', 'Fighting', 'Simulation', 'Party', 'Card', 
#        'Productivity', 'Puzzle', 'Educational', 'Casino', 'RPG', 'Board', 'Other', 'Pinball', 'Platformer', 
#        'Hunting', 'Action', 'Music', 'Compilation', 'Wrestling', 'Trivia'
#    );
#""")
#
# conn.commit()

In [14]:
cur.execute("""
    CREATE TYPE genre_enum AS ENUM (
    'Action',
    'Action, Adventure',
    'Action, Compilation',
    'Action, Editor',
    'Action, Platformer',
    'Action, Puzzle',
    'Action, RPG',
    'Action, Simulation',
    'Action, Strategy',
    'Adult, Card',
    'Adventure',
    'Adventure, Adult',
    'Adventure, Adventure',
    'Adventure, Compilation',
    'Adventure, Episodic',
    'Adventure, Platformer',
    'Adventure, RPG',
    'Baseball',
    'Battle',
    'Board',
    'Board, Compilation',
    'Card',
    'Card, Battle',
    'Card, Compilation',
    'Card, RPG',
    'Casino',
    'Compilation',
    'Compilation, Compilation',
    'Compilation, RPG',
    'Educational',
    'Educational, Action',
    'Educational, Adventure',
    'Educational, Card',
    'Educational, Productivity',
    'Educational, Puzzle',
    'Educational, Simulation',
    'Educational, Trivia',
    'Fighting',
    'Fighting, Action',
    'Fighting, Adventure',
    'Fighting, Compilation',
    'Fighting, RPG',
    'Fighting, Simulation',
    'Flight',
    'Flight, Action',
    'Flight, Racing',
    'Flight, Simulation',
    'Hardware',
    'Hunting',
    'Hunting, Action',
    'Hunting, Simulation',
    'Music',
    'Music, Action',
    'Music, Adventure',
    'Music, Compilation',
    'Music, Editor',
    'Music, RPG',
    'Other',
    'Other, Action',
    'Other, Adventure',
    'Party',
    'Pinball',
    'Pinball, Compilation',
    'Platformer',
    'Platformer, Action',
    'Platformer, Adventure',
    'Productivity',
    'Productivity, Action',
    'Puzzle',
    'Puzzle, Action',
    'Puzzle, Adventure',
    'Puzzle, Compilation',
    'Puzzle, Platformer',
    'Puzzle, RPG',
    'Puzzle, Word Game',
    'Racing',
    'Racing, Action',
    'Racing, Compilation',
    'Racing, Editor',
    'Racing, Shooter',
    'Racing, Simulation',
    'RPG',
    'RPG, Action',
    'RPG, Compilation',
    'RPG, Editor',
    'RPG, Simulation',
    'Shooter',
    'Shooter, Adventure',
    'Shooter, First-Person',
    'Shooter, Platformer',
    'Shooter, RPG',
    'Simulation',
    'Simulation, Adventure',
    'Sports',
    'Sports, Action',
    'Sports, Baseball',
    'Sports, Compilation',
    'Sports, Editor',
    'Sports, Fighting',
    'Sports, Golf',
    'Sports, Other',
    'Sports, Party',
    'Sports, Racing',
    'Sports, Simulation',
    'Strategy',
    'Strategy, Compilation',
    'Strategy, RPG',
    'Strategy, Simulation',
    'Trivia',
    'Virtual Pet',
    'Wrestling',
    'Wrestling, Simulation'
);
""")

conn.commit()

In [15]:
cur.execute("""
    CREATE TYPE platform_enum AS ENUM (
    'PC', 'Game Boy', 'Sega CD', 'Saturn', 'DVD / HD Video Game', 'Nintendo DSi', 
    'Arcade', 'Wii U', 'Lynx', 'Super NES', 'WonderSwan Color', 'TurboGrafx-CD', 
    'Windows Phone', 'TurboGrafx-16', 'N-Gage', 'Xbox One', 'Atari 2600', 
    'Pocket PC', 'Vectrex', 'Nintendo DS', 'Wireless', 'Ouya', 'Nintendo 64DD', 
    'Atari 5200', 'PlayStation 4', 'GameCube', 'Android', 'Wii', 'Game Boy Color', 
    'PlayStation 2', 'New Nintendo 3DS', 'Linux', 'Dreamcast VMU', 'Game Boy Advance', 
    'Windows Surface', 'Genesis', 'Xbox 360', 'Macintosh', 'Web Games', 'Nintendo 3DS', 'iPhone', 
    'SteamOS', 'Commodore 64/128', 'Dreamcast', 'PlayStation 3', 'NES', 'NeoGeo Pocket Color', 
    'Game.Com', 'PlayStation Portable', 'Master System', 'Sega 32X', 'NeoGeo', 'WonderSwan', 'iPad', 
    'Nintendo 64', 'PlayStation Vita', 'Xbox', 'iPod', 'PlayStation');
""")

conn.commit()

Replace empty genre values with NULL values

In [16]:
cur.execute("""
    UPDATE ign_reviews
    SET genre = NULL
    WHERE TRIM(genre) = '';
""")

conn.commit()

In [17]:
cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN platform TYPE platform_enum
    USING platform::platform_enum;
""")    


cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN genre TYPE genre_enum
    USING genre::genre_enum;
""")    

conn.commit()

 Let's make same change for the `url` and `title` column as we made the same observations with `score_phrase` column about the text datatype

In [18]:
cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN title TYPE varchar(200);
""")    

cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN url TYPE varchar(200);
""")   

conn.commit()

The `editors_choice` column contains only `y` and `n` values, which represent whether a game was selected as an editor's choice. Since PostgreSQL's `BOOLEAN` type accepts `y` and `n` as valid representations of `TRUE` and `FALSE`, we can convert this column from `TEXT` to `BOOLEAN`. This makes the data more meaningful and easier to query.

In [22]:
cur.execute("""
    ALTER TABLE ign_reviews
    ALTER COLUMN editors_choice TYPE boolean
    USING CAST(editors_choice AS boolean);
""")

conn.commit()

In [23]:
sql_query = """
SELECT 
    column_name,
    data_type
FROM information_schema.columns
WHERE table_name = 'ign_reviews'  AND column_name='editors_choice'
ORDER BY ordinal_position;
"""

cur.execute(sql_query)

rows = cur.fetchall()

# getting columns name
columns = [desc[0] for desc in cur.description]

# creating the dataframe
df_schema = pd.DataFrame(rows, columns=columns)

# Render result as a dataframe
display(df_schema)

,column_name,data_type
0,editors_choice,boolean


### Converting the Release Date Columns

The `release_year`, `release_month`, and `release_day` columns store the release date separately. This makes date-based queries more complicated.

For example, to find games released after April 2011, we would need to use several conditions:

```sql
SELECT *
FROM ign_reviews
WHERE (release_year > 2011)
   OR (release_year = 2011 AND release_month > 4);

In [25]:
# Creating release_date column

cur.execute("""
    ALTER TABLE ign_reviews
    ADD COLUMN release_date date;
""")

conn.commit()

In [26]:
# filing the relase_date_column

cur.execute("""
    UPDATE ign_reviews
    SET release_date = make_date(
        release_year,
        release_month,
        release_day
    );
""")


In [28]:
# Deleting unecessaring column

cur.execute("""
    ALTER TABLE ign_reviews
    DROP COLUMN release_year,    
    DROP COLUMN release_month,    
    DROP COLUMN release_day;
""")

conn.commit()